** Check for DB Backup Daily **

In [9]:
import socket
#import slack
import configparser
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

In [10]:
config_file = "../guestbook_secrets.ini"
print(config_file)

config = configparser.ConfigParser()
config.read(config_file, encoding="utf-8")
print(config.sections())

_SERVER_ROLE = config['DEFAULT']['role']

_SLACK_TOKEN = config['slack'].get('token')
#print(_SLACK_TOKEN)
 

../guestbook_secrets.ini
['slack', 'guestbookdb']


In [11]:
def postMessageToSlack(message_text):    
  channel_id = "opphouse"

  client = WebClient(token=_SLACK_TOKEN)

  try:
    response = client.chat_postMessage(
        channel=channel_id,
        text=message_text
    )
    print(f"Message sent: {response['ts']}")
  except SlackApiError as e:
    print(f"Error sending message: {e.response['error']}")

In [12]:
import os
import pandas as pd
from datetime import datetime
import glob

if (_SERVER_ROLE == 'STANDBY'):
  now = datetime.now()
  backup_filename = "{}{}".format(now.strftime("%Y_%m_%d"),"__*.backup")
  #print(backup_filename)

  path = "/home/gbadmin/DB_Backups"

  fqfn = "{}/{}".format(path,backup_filename)
  print(fqfn)

  backup_files = glob.glob(fqfn)
  print(backup_files)
  #2025_08_07__15_00_01.backup
  if(len(backup_files)>0):
    print("Backup found")
    file_size = os.path.getsize(backup_files[0])
    message = f"The size of '{backup_files[0]}' is: {file_size} bytes"
    print(message)
    postMessageToSlack(message)
  else:
    message = f"The backup file for today was not found."
    print(message)
    postMessageToSlack(message)

else:
  print("Server is in the {} role, no action has been taken.".format(_SERVER_ROLE))

print("Done!")

Server is in the ACTIVE role, no action has been taken.
Done!
